In [1]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [2]:
builder = (SparkSession.builder
           .appName("create-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")   
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-897c9a9f-6065-4907-be08-67f9d1c5618b;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (256ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (42ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (82ms)
:: resolution report :: resolve 1481ms :: artifac

In [3]:
get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

In [4]:
%%sparksql
CREATE OR REPLACE TABLE default.netflix_titles (
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    date_added STRING,
    release_year STRING,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING 
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/netflix_titles';

In [5]:
import os
print(os.getcwd())  # Prints the current working directory
print(os.path.exists("../data/netflix_titles.csv"))  # Checks if the file exists

/home/jovyan/work/Chapter03
True


In [8]:
import os

# Specify the directory path
directory_path = '/opt/workspace'

# List and print all files in the specified directory
files = os.listdir(directory_path)
print("Files in the directory:", files)


Files in the directory: ['nobel_prizes.json', 'Reviews.csv', 'titles.csv', 'netflix_titles.csv', 'Online_Retail.csv', 'netflix_titles_batch_2.csv', 'recipes.parquet', 'Credit Card', 'nobel_prizes.xml', 'partitioned_recipes', 'Stanford Question Answering Dataset.json', 'data']


In [9]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .load("/opt/workspace/netflix_titles.csv"))


In [ ]:
df = (spark.read
      .format("csv")
      .option("header", "true")
      .load("data/netflix_titles.csv"))


In [ ]:
df.printSchema()

In [ ]:
df.write.format("delta").mode("overwrite").saveAsTable("default.netflix_titles")

In [ ]:
%%sparksql 
SELECT * FROM default.netflix_titles LIMIT 3;

In [ ]:
spark.stop()